In [1]:
import pandas as pd
import torch
from utils import loadVocab, START_TAG, END_TAG, UNK_TAG,vocab_path

In [2]:
#Import module from CDCR folder
import sys
sys.path.append(r'../../CDCR/data')

import dataset

In [3]:
ecbplus_config = dataset.DatasetConfig(data_folder="../../CDCR/data/ECBplus-prep/output_data/",mention_type="entity_mentions",use_gold_mentions=True)
fcc_config = dataset.DatasetConfig(data_folder="../../CDCR/data/FCC-prep/output_data_FCC/",mention_type="event_mentions",use_gold_mentions=True)

In [4]:
from collections import namedtuple
import collections
def temp_read_conll_file(filename):
    Token = namedtuple("Token", "conll_doc_key sentence_id token_id token_str coref_chain")

    current_document_id = ''
    current_document_list = []
    dict_documents = {}
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            
            if line.startswith("#begin") or line == '':
                continue  # Skip the header line

            elif line.startswith("#end") or line == '':
                current_document_id = current_document_list[0].conll_doc_key
                dict_documents[current_document_id] = current_document_list
                current_document_list = []
            else:
                parts = line.split('\t')
                token = Token(parts[0],int(parts[1]),int(parts[2]),parts[3],
                              parts[4].replace('(','').replace(')','')  )

                current_document_list.append(token)

    topics = []
    for key in dict_documents.keys():
        topics.append(key.split('/')[0])
    topics = list(set(topics))

    return dict_documents, topics

In [5]:
def temp_make_dict_of_sentences(documents):
    doc_dict = {}
    for doc, tokens in documents.items():
        dict = collections.defaultdict(list)
        for token in tokens:

            dict[token.sentence_id].append([token.token_id, token.sentence_id, 
                                            token.token_str, token.coref_chain])
        doc_dict[doc] = dict

    return doc_dict

In [6]:
import json
def get_mentions(mentions_path,topics):
    with open(mentions_path, 'r') as f:
        mentions = json.load(f)
    #Filter mentions that belong to the topics
    mentions = [mention for mention in mentions if mention['topic_id'] in topics]
    return mentions


In [7]:
fcc_t_config = dataset.DatasetConfig(data_folder="../../CDCR/data/FCC-prep/output_data_FCC-T/",mention_type="entity_mentions_attr",use_gold_mentions=True)
gvc_config = dataset.DatasetConfig(data_folder="../../CDCR/data/GVC-prep/output_data/",mention_type="event_mentions",use_gold_mentions=True)

meantime_config = dataset.DatasetConfig(data_folder="../../CDCR/data/MEANTIME-prep/output_data/",mention_type="event_mentions",use_gold_mentions=True)
newswcl50_config = dataset.DatasetConfig(data_folder="../../CDCR/data/NewsWCL50-prep/output_data/",mention_type="event_mentions",use_gold_mentions=True)

np4e_config = dataset.DatasetConfig(data_folder="../../CDCR/data/NP4E-prep/output_data/",mention_type="entity_mentions",use_gold_mentions=True)
nident_config = dataset.DatasetConfig(data_folder="../../CDCR/data/NiDENT-prep/output_data/",mention_type="entity_mentions",use_gold_mentions=True)

In [8]:
data_loaded = dataset.CrossEncoderDatasetFull(config=nident_config,split_name='train')

topics keys dict_keys(['0_bomb_explosion_kidnap'])


/home/sago/Documents/GitHub/CDCR_lexical_diversity/MTL/PGKPR_adapted/../../CDCR/data/dataset.py:111: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels,dtype=torch.float)


first torch.Size([25151778])
second torch.Size([25151778])
labels torch.Size([25151778])


In [9]:
len(data_loaded.instances)

510164

In [10]:
ecbplus_train = dataset.CrossEncoderDatasetFull(config=ecbplus_config,split_name='train')
ecbplus_test = dataset.CrossEncoderDatasetFull(config=ecbplus_config,split_name='test')
ecbplus_val = dataset.CrossEncoderDatasetFull(config=ecbplus_config,split_name='val')

topics keys dict_keys(['1', '10', '11', '13', '14', '16', '19', '20', '22', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '4', '6', '7', '8', '9'])
first torch.Size([833372])
second torch.Size([833372])
labels torch.Size([833372])
topics keys dict_keys(['36', '37', '38', '39', '40', '41', '42', '43', '44', '45'])
first torch.Size([380551])
second torch.Size([380551])
labels torch.Size([380551])
topics keys dict_keys(['12', '18', '2', '21', '23', '34', '35', '5'])
first torch.Size([197512])
second torch.Size([197512])
labels torch.Size([197512])


In [11]:
len(ecbplus_train.instances)

240747

In [12]:
#Modification of the original load_text_con method. Adds start and end tags
def load_text_con(cdcr_dataset_split,max_rows=None):
    if max_rows==None:
        max_rows=len(cdcr_dataset_split.instances)
    #Load CDCR data
    
    inputs = [[START_TAG] + x.strip().split(" ") + [END_TAG] for x in cdcr_dataset_split.instances[0:max_rows]] 
    # I could add this to the standard processing. There is unnecessary overhead
    

    # label_with_pos = [[START_TAG] + x + [SEP_TAG] + y + [END_TAG] for x, y in zip(label_sen, label_pos)]
    labels = cdcr_dataset_split.labels[0:max_rows]

    return inputs, labels

In [13]:
#Temp vocab based on MSCOCO. I should use a pre-trained subtokenizer
#loadVocab(vocab_fpath="mscoco.vocab")

In [14]:
def load_data_tensor_con(cdcr_dataset,max_rows, vocabPath, maxlen=300): #  have to adapt the max length become the max length of the input dataset
    token2idx, _ = loadVocab(vocabPath)

    tokenized_inputs = []
    inputs, labels = load_text_con(cdcr_dataset,max_rows)
    padded_length = maxlen
    for x in inputs:
        x = [token2idx.get(t, token2idx[UNK_TAG]) for t in x] #Fills with Unknown tag value if token not found 
        
        x = x + [0] * (padded_length - len(x)) 

        tokenized_inputs.append(x)
    return torch.tensor(tokenized_inputs), labels

In [14]:
#Check if lengths from both methods differ
# I should not pad here but when I create the dataloader. I should use the maximum length 
# of all the datasets and splits that I use

In [15]:
len(ecbplus_train.instances)

240747

In [16]:
ecbplus_train.labels[0]

tensor(1.)

In [17]:
ecbplus_train_tokenized_inputs, ecbplus_train_labels = load_data_tensor_con(cdcr_dataset=ecbplus_train,max_rows=None,vocabPath=vocab_path)
ecbplus_test_tokenized_inputs, ecbplus_test_labels = load_data_tensor_con(cdcr_dataset=ecbplus_test,max_rows=None,vocabPath=vocab_path)
ecbplus_val_tokenized_inputs, ecbplus_val_labels = load_data_tensor_con(cdcr_dataset=ecbplus_test,max_rows=None,vocabPath=vocab_path)


In [18]:
def get_max_sequence_length(tensor):
    # Calculate the actual sequence lengths by counting non-padding values
    actual_seq_lengths = (tensor != 0).sum(dim=1)
    # Find the maximum sequence length and its index
    max_seq_length, max_seq_index = actual_seq_lengths.max(dim=0)
    max_seq_length = max_seq_length.item()
    max_seq_index = max_seq_index.item()
    return max_seq_length, max_seq_index

In [19]:
max_length, index = get_max_sequence_length(ecbplus_train_tokenized_inputs)
max_length

296

In [20]:
ecbplus_train.instances[index]

"Phil Ford 's school mark of 2,290 points set in 1978 held strong amongst a plethora of talented Tar Heels over the years , but [START] Hansbrough [END] , who came back for his senior season after being crowned the National Player of the Year , moved into the top spot with a tough leaner in the lane with two Purple Aces defenders hanging on him with 7:42 left in the first half . Phil Ford 's school mark of 2,290 points set in 1978 held strong amongst a plethora of talented Tar Heels over the years , but Hansbrough , who came back for his senior season after being crowned the National Player of the Year , moved into the top spot with a tough leaner in the lane with two Purple Aces defenders hanging on him with 7:42 left in the first half . [SEP] Phil Ford 's school mark of 2,290 points set in 1978 held strong amongst a plethora of talented Tar Heels over the years , but Hansbrough , who came back for his senior season after being crowned the [START] National Player of the Year [END] , m

In [21]:

torch.save(ecbplus_train_tokenized_inputs, 'train_inputs.pt')
torch.save(ecbplus_train_labels, 'train_labels.pt')

torch.save(ecbplus_test_tokenized_inputs, 'test_inputs.pt')
torch.save(ecbplus_test_labels, 'test_labels.pt')

torch.save(ecbplus_val_tokenized_inputs, 'val_inputs.pt')
torch.save(ecbplus_val_labels, 'val_labels.pt')



In [ ]:

# (Zeng et al., 2020) Suggest oversampling positive pairs and undersampling negative pairs